<a href="https://colab.research.google.com/github/anonpc/LLM_WITH_KNOWLEDGES/blob/main/Chapter_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 2: Our First Model

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES=True

In [7]:
import os
import sys
import pandas as pd
import itertools
import shutil
import requests
from urllib.parse import urlparse

# Убедитесь, что все необходимые пакеты установлены
from requests.exceptions import HTTPError, RequestException

classes = ["cat", "fish"]
set_types = ["train", "test", "val"]

def download_image(url, klass, data_type):
    basename = os.path.basename(urlparse(url).path)
    filename = os.path.join(data_type, klass, basename)

    if not os.path.exists(filename):
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()  # Пытаемся получить исключение, если запрос завершился с ошибкой

            with open(filename, "wb") as out_file:
                shutil.copyfileobj(response.raw, out_file)
            print(f"Downloaded {url} to {filename}")
        except HTTPError as http_err:
            print(f"HTTP error occurred: {http_err} for {url}")
        except RequestException as req_err:
            print(f"Request error occurred: {req_err} for {url}")
        except Exception as e:
            print(f"Unexpected error: {e} for {url}")

if __name__ == "__main__":
    csv_path = "/content/images.csv"

    if not os.path.exists(csv_path):
        print(f"Error: can't find {csv_path}!")
        sys.exit(0)

    try:
        imagesDF = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        print("Error: The CSV file is empty.")
        sys.exit(1)
    except pd.errors.ParserError:
        print("Error: There was a problem parsing the CSV file.")
        sys.exit(1)

    print(f"CSV file read successfully. Columns: {imagesDF.columns.tolist()}")

    for set_type, klass in itertools.product(set_types, classes):
        path = os.path.join(set_type, klass)
        if not os.path.exists(path):
            print(f"Creating directory {path}")
            os.makedirs(path)

    print(f"Downloading {len(imagesDF)} images")

    for url, klass, data_type in zip(imagesDF["url"], imagesDF["class"], imagesDF["type"]):
        download_image(url, klass, data_type)

    print("Download process completed.")
    sys.exit(0)


CSV file read successfully. Columns: ['url', 'class', 'type']
Creating directory train/cat
Creating directory train/fish
Creating directory test/cat
Creating directory test/fish
Creating directory val/cat
Creating directory val/fish
Downloaded http://farm2.static.flickr.com/1245/1259825348_6a2aa94e8d.jpg to train/cat/1259825348_6a2aa94e8d.jpg
Downloaded http://farm2.static.flickr.com/1080/1029412358_7ee17550fc.jpg to train/cat/1029412358_7ee17550fc.jpg
Downloaded http://farm1.static.flickr.com/196/443645811_8c4bb1af50.jpg to train/cat/443645811_8c4bb1af50.jpg
Downloaded http://farm2.static.flickr.com/1201/1285591549_593ca7cf6a.jpg to train/cat/1285591549_593ca7cf6a.jpg
Downloaded http://farm3.static.flickr.com/2002/1760479467_068432dd3f.jpg to train/cat/1760479467_068432dd3f.jpg
Downloaded http://farm2.static.flickr.com/1411/649938210_e9dcbde5ea.jpg to train/cat/649938210_e9dcbde5ea.jpg
Downloaded http://farm2.static.flickr.com/1423/918775832_9ecfd414b8.jpg to train/cat/918775832_9ecfd

SystemExit: 0

/usr/local/lib/python3.10/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Setting up DataLoaders

We'll use the built-in dataset of `torchvision.datasets.ImageFolder` to quickly set up some dataloaders of downloaded cat and fish images.

`check_image`  is a quick little function that is passed to the `is_valid_file` parameter in the ImageFolder and will do a sanity check to make sure PIL can actually open the file. We're going to use this in lieu of cleaning up the downloaded dataset.


In [8]:
def check_image(path):
    try:
        im = Image.open(path)
        return True
    except:
        return False

Set up the transforms for every image:

* Resize to 64x64
* Convert to tensor
* Normalize using ImageNet mean & std


In [9]:
img_transforms = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225] )
    ])



In [11]:
train_data_path = "./train/"
train_data = torchvision.datasets.ImageFolder(root=train_data_path,transform=img_transforms, is_valid_file=check_image)

In [12]:
val_data_path = "./val/"
val_data = torchvision.datasets.ImageFolder(root=val_data_path,transform=img_transforms, is_valid_file=check_image)

In [13]:
test_data_path = "./test/"
test_data = torchvision.datasets.ImageFolder(root=test_data_path,transform=img_transforms, is_valid_file=check_image)

In [14]:
batch_size=64

In [15]:
train_data_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size)
val_data_loader  = torch.utils.data.DataLoader(val_data, batch_size=batch_size)
test_data_loader  = torch.utils.data.DataLoader(test_data, batch_size=batch_size)

## Our First Model, SimpleNet

SimpleNet is a very simple combination of three Linear layers and ReLu activations between them. Note that as we don't do a `softmax()` in our `forward()`, we will need to make sure we do it in our training function during the validation phase.

In [16]:
class SimpleNet(nn.Module):

    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(12288, 84)
        self.fc2 = nn.Linear(84, 50)
        self.fc3 = nn.Linear(50,2)

    def forward(self, x):
        x = x.view(-1, 12288)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [17]:
simplenet = SimpleNet()

## Create an optimizer

Here, we're just using Adam as our optimizer with a learning rate of 0.001.

In [18]:
optimizer = optim.Adam(simplenet.parameters(), lr=0.001)

## Copy the model to GPU

Copy the model to the GPU if available.

In [19]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

simplenet.to(device)

SimpleNet(
  (fc1): Linear(in_features=12288, out_features=84, bias=True)
  (fc2): Linear(in_features=84, out_features=50, bias=True)
  (fc3): Linear(in_features=50, out_features=2, bias=True)
)

## Training

Trains the model, copying batches to the GPU if required, calculating losses, optimizing the network and perform validation for each epoch.

In [20]:
def train(model, optimizer, loss_fn, train_loader, val_loader, epochs=20, device="cpu"):
    for epoch in range(1, epochs+1):
        training_loss = 0.0
        valid_loss = 0.0
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            inputs, targets = batch
            inputs = inputs.to(device)
            targets = targets.to(device)
            output = model(inputs)
            loss = loss_fn(output, targets)
            loss.backward()
            optimizer.step()
            training_loss += loss.data.item() * inputs.size(0)
        training_loss /= len(train_loader.dataset)

        model.eval()
        num_correct = 0
        num_examples = 0
        for batch in val_loader:
            inputs, targets = batch
            inputs = inputs.to(device)
            output = model(inputs)
            targets = targets.to(device)
            loss = loss_fn(output,targets)
            valid_loss += loss.data.item() * inputs.size(0)
            correct = torch.eq(torch.max(F.softmax(output, dim=1), dim=1)[1], targets)
            num_correct += torch.sum(correct).item()
            num_examples += correct.shape[0]
        valid_loss /= len(val_loader.dataset)

        print('Epoch: {}, Training Loss: {:.2f}, Validation Loss: {:.2f}, accuracy = {:.2f}'.format(epoch, training_loss,
        valid_loss, num_correct / num_examples))

In [21]:
train(simplenet, optimizer,torch.nn.CrossEntropyLoss(), train_data_loader,val_data_loader, epochs=5, device=device)

Epoch: 1, Training Loss: 3.76, Validation Loss: 1.70, accuracy = 0.48
Epoch: 2, Training Loss: 1.92, Validation Loss: 1.65, accuracy = 0.47
Epoch: 3, Training Loss: 1.28, Validation Loss: 0.92, accuracy = 0.65
Epoch: 4, Training Loss: 0.63, Validation Loss: 1.08, accuracy = 0.65
Epoch: 5, Training Loss: 0.58, Validation Loss: 0.72, accuracy = 0.74


## Making predictions

Labels are in alphanumeric order, so `cat` will be 0, `fish` will be 1. We'll need to transform the image and also make sure that the resulting tensor is copied to the appropriate device before applying our model to it.

In [22]:
labels = ['cat','fish']

img = Image.open("./val/fish/100_1422.JPG")
img = img_transforms(img).to(device)
img = torch.unsqueeze(img, 0)

simplenet.eval()
prediction = F.softmax(simplenet(img), dim=1)
prediction = prediction.argmax()
print(labels[prediction])

fish


## Saving Models

We can either save the entire model using `save` or just the parameters using `state_dict`. Using the latter is normally preferable, as it allows you to reuse parameters even if the model's structure changes (or apply parameters from one model to another).

In [23]:
torch.save(simplenet, "/tmp/simplenet")
simplenet = torch.load("/tmp/simplenet")


In [24]:
torch.save(simplenet.state_dict(), "/tmp/simplenet")
simplenet = SimpleNet()
simplenet_state_dict = torch.load("/tmp/simplenet")
simplenet.load_state_dict(simplenet_state_dict)

<All keys matched successfully>